In [11]:
import os
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

BASE_PATH = "/kaggle/input/datasets/ashishjangra27/face-mask-12k-images-dataset/Face Mask Dataset/Train"

mask_path = os.path.join(BASE_PATH, "WithMask")
nomask_path = os.path.join(BASE_PATH, "WithoutMask")

mask_imgs = os.listdir(mask_path)
nomask_imgs = os.listdir(nomask_path)

images = []
labels = []

# With Mask → label 1
for img in mask_imgs:
    img = Image.open(os.path.join(mask_path, img)).convert("L")
    img = img.resize((64,64))
    images.append(np.array(img)/255.0)
    labels.append(1)

# Without Mask → label 0
for img in nomask_imgs:
    img = Image.open(os.path.join(nomask_path, img)).convert("L")
    img = img.resize((64,64))
    images.append(np.array(img)/255.0)
    labels.append(0)

# Convert to numpy
X = np.array(images)
y = np.array(labels)

# Add channel dimension
X = X.reshape(-1, 1, 64, 64)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        self.conv = nn.Conv2d(1, 16, 3)
        self.pool = nn.MaxPool2d(2,2)
        self.fc = nn.Linear(16 * 31 * 31, 1)
        
    def forward(self, x):
        x = self.conv(x)
        x = torch.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)  # No sigmoid here
        return x

model = SimpleCNN()


criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0
    
    for xb, yb in train_loader:
        outputs = model(xb).squeeze()
        loss = criterion(outputs, yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

Epoch [1/10], Loss: 88.6972
Epoch [2/10], Loss: 44.2605
Epoch [3/10], Loss: 34.0142
Epoch [4/10], Loss: 28.8727
Epoch [5/10], Loss: 24.1808
Epoch [6/10], Loss: 21.8637
Epoch [7/10], Loss: 19.1310
Epoch [8/10], Loss: 17.6957
Epoch [9/10], Loss: 16.1487
Epoch [10/10], Loss: 15.8805
